# 03 - Player counts table

- First look at `STEAM_DB.STEAM_RAW.PLAYER_COUNTS`.
- dlt loads it from the `GetNumberOfCurrentPlayers` endpoint, which works without an API key.

- One row is one game at one moment, with the number of players Steam says are in the game right now.
- Unlike the other two tables, `collected_at` is set per row (right after Steam answers), not once per run. A player count is a single moment, and a run takes about 10 minutes.

- The data is one load from 2026-10-27 (load id `1793094197.9862032`), 81 Paradox games. The run started at about 10:43 Stockholm time.
- The notebook only runs SELECT and DESCRIBE, so it changes nothing in Snowflake.

- Limits: there is only one snapshot, so no history yet. A player count is one moment and depends on the time of day. For small games, 1 or 2 players is noise. Steam gives no sales numbers, so players are only a stand-in for how active a game is, and we can show correlation, not causation.

## 0. Connect to Snowflake

Same connection as notebooks 01 and 02. It reads the settings from `.env`, logs in as `STEAM_LOADER` and prints no secrets. Times are shown in UTC (Stockholm is UTC+1 now). Pick the project's `.venv` as the kernel.

In [14]:
import os
from pathlib import Path

import pandas as pd
import snowflake.connector

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

In [ ]:
def find_repo_root() -> Path:
    """Walk up from the notebook's folder until we find pyproject.toml."""
    here = Path.cwd().resolve()
    return next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())


def load_env_file(path: Path) -> None:
    """Copy KEY=value lines from a .env file into os.environ (never prints values)."""
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("\"'"))


load_env_file(find_repo_root() / ".env")

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_LOADER_USER"],
    private_key_file=os.environ["SNOWFLAKE_LOADER_PRIVATE_KEY_PATH"],
    role=os.environ["SNOWFLAKE_LOADER_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema="STEAM_RAW",
    session_parameters={"TIMEZONE": "UTC"},
)


def query(sql: str) -> pd.DataFrame:
    """Run one read-only statement and return the result as a DataFrame."""
    with conn.cursor() as cursor:
        cursor.execute(sql)
        columns = [column[0] for column in cursor.description]
        return pd.DataFrame(cursor.fetchall(), columns=columns)

In [16]:
# Sanity check: we should be the loader role, in STEAM_DB.STEAM_RAW.
query("""
SELECT CURRENT_ROLE() AS role, CURRENT_WAREHOUSE() AS warehouse,
       CURRENT_DATABASE() AS database, CURRENT_SCHEMA() AS schema
""")

,ROLE,WAREHOUSE,DATABASE,SCHEMA
0,STEAM_LOADER,STEAM_WH,STEAM_DB,STEAM_RAW


## 1. Rows per load

- Every run adds one row per game. Each row has its own `collected_at`, so we group by the dlt load id to see a run.

**💡Insights:**
- One load, 81 rows, 81 different games (checked in Snowsight on 2026-10-27).
- The load id is `1793094197.9862032`, the same as in the log.
- The first row was collected at 09:43:19 UTC and the last at 09:53:27 UTC (10:43 to 10:53 Stockholm). So one run takes about 10 minutes, and the rows do not share one time.

In [17]:
query("""
SELECT _dlt_load_id, COUNT(*) AS rows_loaded, COUNT(DISTINCT steam_appid) AS distinct_games,
       MIN(collected_at) AS first_row, MAX(collected_at) AS last_row
FROM player_counts
GROUP BY _dlt_load_id
ORDER BY _dlt_load_id
""")

,_DLT_LOAD_ID,ROWS_LOADED,DISTINCT_GAMES,FIRST_ROW,LAST_ROW
0,1793094197.9862032,81,81,2026-10-27 09:43:19.528098+00:00,2026-10-27 09:53:27.599330+00:00


## 2. Duplicates

A game should appear only once per run. This lists any game that shows up twice in the same load.

**💡Insights:**
- Section 1 already shows 81 rows for 81 games in one load, so no duplicates are expected.

In [18]:
query("""
SELECT steam_appid, _dlt_load_id, COUNT(*) AS copies
FROM player_counts
GROUP BY steam_appid, _dlt_load_id
HAVING COUNT(*) > 1
""")

,STEAM_APPID,_DLT_LOAD_ID,COPIES


## 3. Column types

**💡Insights:**
- `COLLECTED_AT` is `TIMESTAMP_LTZ(9)`, the same type as in the other two tables.
- `STEAM_APPID`, `PLAYER_COUNT` and `RESULT` are `NUMBER(19,0)`.
- The `columns` hint on the resource sets `PLAYER_COUNT` and `RESULT` as integers, so a batch with NULL counts cannot confuse dlt's type guess.
- dlt added `_DLT_LOAD_ID` and `_DLT_ID`. Every other column can be NULL.

In [19]:
query("DESCRIBE TABLE player_counts")[["name", "type", "null?"]]

,name,type,null?
0,COLLECTED_AT,TIMESTAMP_LTZ(9),Y
1,PLAYER_COUNT,"NUMBER(19,0)",Y
2,RESULT,"NUMBER(19,0)",Y
3,STEAM_APPID,"NUMBER(19,0)",Y
4,_DLT_LOAD_ID,VARCHAR(16777216),N
5,_DLT_ID,VARCHAR(16777216),N


## 4. Result codes

- Steam sends a `result` code with every answer. It tells us if Steam had data for the game.
- This query counts the rows per code and separates zero players from the rest.

**💡Insights:**

| `result` | What Steam sent | `player_count` | Games |
|---:|---|---|---:|
| 1 | A normal answer (HTTP 200) | A number. It can be 0. | 77 |
| 42 | "No data" (HTTP 404) | Missing, so NULL in our table | 4 |

How to read a row:

| `player_count` | `result` | What it means |
|---|---|---|
| 7,992 | 1 | That many players right now |
| 0 | 1 | A real zero. Steam knows the game and nobody is playing |
| NULL | 42 | Steam has no data. Not the same as zero |

- Both codes (1 and 42) were seen in the probe and again in the full run.
- Checked in Snowsight on 2026-10-27: `result` is 1 for all 77 rows with a count (64 above zero, 13 with 0 players) and 42 for the 4 rows without a count.
- We believe 42 means "no match", but that is from memory and not checked. In this project we only say "Steam has no data".

In [20]:
query("""
SELECT result, player_count = 0 AS is_zero, COUNT(*) AS games
FROM player_counts
GROUP BY result, is_zero
ORDER BY result, is_zero
""")

,RESULT,IS_ZERO,GAMES
0,1,False,64
1,1,True,13
2,42,None,4


## 5. Games without data

- When Steam has no data for a game, it answers HTTP 404 with `result = 42` and no `player_count`.
- We keep these games as a row with a NULL count.

**💡Insights:**

| Game | appid | Release date | `coming_soon` | `player_count` | `result` |
|---|---:|---|:---:|---|---:|
| Prison Architect 2 | 1257610 | To be announced | True | NULL | 42 |
| LEGO Skylines | 1573550 | Coming soon | True | NULL | 42 |
| Guardians of the Wild Sky | 1579420 | Coming soon | True | NULL | 42 |
| Afterworld | 4920230 | To be announced | True | NULL | 42 |

- Only 4 rows have no count, and all 4 are unreleased games.
- They are the same 4 games that have no reviews in notebook 02.
- No released game came back without data in this run.

| Question | Answer |
|---|---|
| Does `result = 42` mean "unreleased"? | No. In the probe a fake appid gave the same answer (404, result 42). |
| How do we know a game is unreleased? | Use `coming_soon` from the `games` table. |

In [21]:
query("""
SELECT p.steam_appid, g.name, g.release_date_text, g.coming_soon, p.player_count, p.result
FROM player_counts p
LEFT JOIN (
    SELECT *
    FROM games
    QUALIFY ROW_NUMBER() OVER (PARTITION BY steam_appid ORDER BY collected_at DESC) = 1
) g ON g.steam_appid = p.steam_appid
WHERE p.player_count IS NULL
ORDER BY p.steam_appid
""")

,STEAM_APPID,NAME,RELEASE_DATE_TEXT,COMING_SOON,PLAYER_COUNT,RESULT
0,1257610,Prison Architect 2,To be announced,True,None,42
1,1573550,LEGO® Skylines,Coming soon,True,None,42
2,1579420,Guardians of the Wild Sky,Coming soon,True,None,42
3,4920230,Afterworld,To be announced,True,None,42


## 6. Games with zero players

- A zero is a real answer: Steam sent `player_count = 0`. It is not the same as "no data" (section 5).

**💡Insights:**

| | Games |
|---|---:|
| With a count | 77 |
| of which 0 players | 13 |
| Without data | 4 |

- 13 of the 77 games with a count show 0 players at that moment.
- All 13 are released, because the only unreleased games are the 4 in section 5.

| Game with 0 players | Reviews (notebook 02) |
|---|---:|
| NECROPOLIS: BRUTAL EDITION | 2,635 |
| Majesty Gold HD | 374 |
| Impire | 259 |
| Sengoku | 250 |
| King Arthur II: The Role-Playing Wargame | 222 |
| Ancient Space | 207 |
| The Kings' Crusade | 67 |
| Premium Pool Arena | 50 |
| Race Arcade | 35 |
| King Arthur: Fallen Champions | 27 |
| Europa Universalis III: Divine Wind | 27 |
| Hearts of Iron III: Semper Fi | 1 |
| Knights of Pen and Paper 2: Free Edition | 0 |

- A game with reviews can still show 0 players. Reviews are all-time totals, players are one moment.
- *Knights of Pen and Paper 2: Free Edition* has 0 players and no reviews. We still do not know why.
- A 0 at one moment does not mean nobody ever plays the game.

In [22]:
query("""
SELECT g.name, g.release_date_text, g.coming_soon, g.is_free, r.total_reviews
FROM player_counts p
LEFT JOIN (
    SELECT *
    FROM games
    QUALIFY ROW_NUMBER() OVER (PARTITION BY steam_appid ORDER BY collected_at DESC) = 1
) g ON g.steam_appid = p.steam_appid
LEFT JOIN (
    SELECT *
    FROM review_summary
    QUALIFY ROW_NUMBER() OVER (PARTITION BY steam_appid ORDER BY collected_at DESC) = 1
) r ON r.steam_appid = p.steam_appid
WHERE p.player_count = 0
ORDER BY r.total_reviews DESC
""")

,NAME,RELEASE_DATE_TEXT,COMING_SOON,IS_FREE,TOTAL_REVIEWS
0,NECROPOLIS: BRUTAL EDITION,"12 Jul, 2016",False,False,2635
1,Majesty Gold HD,"4 Sep, 2009",False,False,374
2,Impire,"14 Feb, 2013",False,False,259
3,Sengoku,"15 Sep, 2011",False,False,250
4,King Arthur II: The Role-Playing Wargame,"27 Jan, 2012",False,False,222
5,Ancient Space,"23 Sep, 2014",False,False,207
6,The Kings' Crusade,"8 Oct, 2010",False,False,67
7,Premium Pool Arena,"14 Apr, 2017",False,False,50
8,Race Arcade,"5 Apr, 2017",False,False,35
9,Europa Universalis III: Divine Wind,"21 Jan, 2011",False,False,27


## 7. Players per game

**💡Insights:**

**Most players right now**

| Game | Players | Reviews (notebook 02) |
|---|---:|---:|
| Hearts of Iron IV | 26,021 | 123,314 |
| Crusader Kings III | 12,393 | 50,404 |
| Transport Fever 3 | 8,468 | 3,067 |
| Stellaris | 8,106 | 76,804 |
| Cities: Skylines II | 6,820 | 34,961 |

- Nine games have more than 3,900 players. The 10th, Age of Wonders 4, has 1,033.
- The top 9 add up to about 83,000. This is my hand sum of the export, not a query.

**Players and reviews do not rank the same**

| Game | Players | Reviews | What it shows |
|---|---:|---:|---|
| Transport Fever 3 | 8,468 | 3,067 | 3rd by players, but 24th by reviews |
| Vampire: The Masquerade - Bloodlines 2 | 41 | 6,682 | Many reviews, few players now |
| NECROPOLIS: BRUTAL EDITION | 0 | 2,635 | Many reviews, nobody playing at that moment |

**The number moves, even for a big game**

| Stellaris, same morning | Players |
|---|---:|
| Probe | 7,859 |
| Test run on 3 apps | 7,992 |
| Full run | 8,106 |

- So one count is a snapshot of one moment, not a stable number.

In [23]:
query("""
SELECT g.name, p.player_count, p.collected_at
FROM player_counts p
LEFT JOIN (
    SELECT steam_appid, name
    FROM games
    QUALIFY ROW_NUMBER() OVER (PARTITION BY steam_appid ORDER BY collected_at DESC) = 1
) g ON g.steam_appid = p.steam_appid
ORDER BY p.player_count DESC NULLS LAST
""")

,NAME,PLAYER_COUNT,COLLECTED_AT
0,Hearts of Iron IV,26021.0,2026-10-27 09:43:34.968238+00:00
1,Crusader Kings III,12393.0,2026-10-27 09:43:19.528098+00:00
2,Transport Fever 3,8468.0,2026-10-27 09:43:27.020015+00:00
3,Stellaris,8106.0,2026-10-27 09:43:42.527838+00:00
4,Cities: Skylines II,6820.0,2026-10-27 09:43:50.012648+00:00
5,Europa Universalis IV,6494.0,2026-10-27 09:44:12.478831+00:00
6,Cities: Skylines,6271.0,2026-10-27 09:44:27.673402+00:00
7,Victoria 3,4405.0,2026-10-27 09:43:57.498356+00:00
8,Europa Universalis V,3910.0,2026-10-27 09:44:19.980886+00:00
9,Age of Wonders 4,1033.0,2026-10-27 09:44:04.933727+00:00


## 8. How many players do games have?

A difference of 1 or 2 players says nothing, so this counts the games by number of players.

**💡Insights:**

| Players | Games |
|---|---:|
| no data | 4 |
| 0 | 13 |
| 1-9 | 19 |
| 10-99 | 25 |
| 100-999 | 10 |
| 1,000-9,999 | 8 |
| 10,000 or more | 2 |

- About 35 of the 77 games with a count have 10 players or fewer (13 with 0 and 22 with 1 to 10). Only 10 games have 1,000 or more.
- For most games this one snapshot is too small and noisy to compare. It works better for ranking the top games.

In [24]:
query("""
SELECT
    CASE
        WHEN player_count IS NULL THEN '0 no data'
        WHEN player_count = 0 THEN '1 zero'
        WHEN player_count < 10 THEN '2 (1-9)'
        WHEN player_count < 100 THEN '3 (10-99)'
        WHEN player_count < 1000 THEN '4 (100-999)'
        WHEN player_count < 10000 THEN '5 (1,000-9,999)'
        ELSE '6 (10,000+)'
    END AS player_volume,
    COUNT(*) AS games
FROM player_counts
GROUP BY player_volume
ORDER BY player_volume
""")

,PLAYER_VOLUME,GAMES
0,0 no data,4
1,1 zero,13
2,2 (1-9),19
3,3 (10-99),25
4,4 (100-999),10
5,"5 (1,000-9,999)",8
6,"6 (10,000+)",2


## 9. First look: players and reviews

- A first look at one of the business questions, using the latest review snapshot.
- Only games with a count and at least 100 reviews. One snapshot can't show cause and effect.

**💡Insights:**
- From sections 6 and 7 we already know that reviews and players do not rank the same.
- No correlation has been calculated.

In [25]:
query("""
SELECT g.name, p.player_count, r.total_reviews,
       ROUND(100 * r.total_positive / NULLIF(r.total_reviews, 0), 1) AS pct_positive
FROM player_counts p
JOIN (
    SELECT *
    FROM review_summary
    QUALIFY ROW_NUMBER() OVER (PARTITION BY steam_appid ORDER BY collected_at DESC) = 1
) r ON r.steam_appid = p.steam_appid
JOIN (
    SELECT steam_appid, name
    FROM games
    QUALIFY ROW_NUMBER() OVER (PARTITION BY steam_appid ORDER BY collected_at DESC) = 1
) g ON g.steam_appid = p.steam_appid
WHERE p.player_count IS NOT NULL AND r.total_reviews >= 100
ORDER BY p.player_count DESC
""")

,NAME,PLAYER_COUNT,TOTAL_REVIEWS,PCT_POSITIVE
0,Hearts of Iron IV,26021,123303,91.0
1,Crusader Kings III,12393,50408,90.4
2,Transport Fever 3,8468,3084,72.6
3,Stellaris,8106,76814,86.1
4,Cities: Skylines II,6820,34968,58.1
5,Europa Universalis IV,6494,47761,87.0
6,Cities: Skylines,6271,100556,92.8
7,Victoria 3,4405,20709,70.1
8,Europa Universalis V,3910,13848,68.1
9,Age of Wonders 4,1033,11172,82.9


## 10. Findings and decisions for dbt staging

- Each finding below says the findings, importance and dbt tasks for each findings.
- These are proposals until we build `feat/dbt-staging`. The SQL snippets are sketches to explain the idea. They are not tested.
- Model and column names (like `stg_steam__player_counts`) are only suggestions.

**Terminologies**

| Schema | Meaning |
|---|---|
| Staging | The first dbt layer. One model per raw table. We rename, cast types and clean. No joins, no business logic. |
| Intermediate | The second layer. Joins tables together and adds logic (for example "latest row per game"). |
| Marts | The last layer. Tables the dashboard reads. |
| Test | A check that dbt runs with `dbt test`. If it finds bad rows, the test fails and dbt shows them. |
| Flag | A True/False column that makes a rule easy to use later (for example `has_player_data`). |
| Source | The raw table in `STEAM_RAW` that dbt reads from. Set up in `feat/dbt-sources`. |

---

### Finding 1: GRAIN: one row per game per load, and a run id

| | |
|---|---|
| Findings | 81 rows, 81 different games, one load (sections 1, 2). `collected_at` is different for every row. |
| Why it matters | If a game appeared twice in one run, counts and averages would be wrong. We also need a way to say "these rows belong to the same run". `collected_at` can't do that, because every row has its own time. |
| What dbt does | In the staging model, keep `steam_appid`, `collected_at`, `player_count`, `result` and rename `_dlt_load_id` to `load_id`. `load_id` is the same for every row of one run. |
| Tests to add | `unique` on the pair (`steam_appid`, `load_id`). This needs the `dbt_utils` package or our own SQL test. We have not checked if the package is installed. `not_null` on `steam_appid`, `collected_at` and `result`. A `relationships` test that every `steam_appid` also exists in `games`. |
| Layer | Staging |

---

### Finding 2: time

| | |
|---|---|
| Findings | `collected_at` is `TIMESTAMP_LTZ(9)` (section 3). It is taken per row, so a run spans about 10 minutes (section 1). `games` and `review_summary` have one `collected_at` per run. |
| Why it matters | A `TIMESTAMP_LTZ` shows in whatever time zone the session uses. The same row can look different in Snowsight and in dbt. Also, because the three tables have different kinds of times, we can't join them on the exact timestamp. It would never match. |
| What dbt does | In staging, convert to UTC on purpose and add two helper columns: the date and the Stockholm hour (so we can compare snapshots taken at the same time of day). In intermediate, join `player_counts` to `games` and `review_summary` on the **date**. |
| Sketch | `convert_timezone('UTC', collected_at) as collected_at_utc`<br>`to_date(convert_timezone('UTC', collected_at)) as collected_date`<br>`hour(convert_timezone('Europe/Stockholm', collected_at)) as collected_hour_local` |
| Careful | Joining on date only works if there is one run per day. That rule is still open (see the list at the end). |
| Layer | Staging (the columns), intermediate (the join) |

---

### Finding 3: NULL means "no data", 0 means "zero players"

| | |
|---|---|
| Findings | 4 games have a NULL count and `result = 42`. 13 games have a real 0 (sections 5, 6). |
| Why it matters | If dbt turns NULL into 0, unreleased games look like games nobody plays. That would pull averages down and give a wrong story. |
| What dbt does | Keep `player_count` exactly as it is. Never use `coalesce(player_count, 0)`. Add a flag `has_player_data` that is true when `result = 1`. |
| Sketch | `player_count,`<br>`(result = 1) as has_player_data` |
| Checked | The section 4 query confirmed it: all 13 zero rows have `result = 1`, so `has_player_data = (result = 1)` holds. |
| Layer | Staging |

---

### Finding 3b: protect against surprises

| | |
|---|---|
| Findings | We have only seen `result` 1 and 42. |
| Why it matters | Steam might send a new code one day, or a count with the wrong code. We want to hear about it instead of loading it silently. |
| What dbt does | An `accepted_values` test on `result` with the values 1 and 42, set to `warn` (the run continues, but we see a warning). Plus our own SQL test that returns the bad rows. The test passes when it returns no rows. |
| Sketch | `select * from {{ ref('stg_steam__player_counts') }}`<br>`where (result = 1 and player_count is null)`<br>`   or (result = 42 and player_count is not null)` |
| Layer | Staging tests |

---

### Finding 4: `result = 42` does not mean "unreleased"

| | |
|---|---|
| Findings | A fake appid gave the same answer as an unreleased game (404, result 42) in the probe (section 5). |
| Why it matters | If dbt used 42 to say "unreleased", a wrong id would be called unreleased too. |
| What dbt does | Take "unreleased" from `coming_soon` in the `games` table. Join to the **latest** `games` row per game. In this notebook we do that with `QUALIFY ROW_NUMBER() OVER (PARTITION BY steam_appid ORDER BY collected_at DESC) = 1`. dbt can use the same. |
| Layer | Intermediate (the join), marts (the final flag) |

---

### Finding 5: most games have very few players

| | |
|---|---|
| Findings | About 35 of 77 games have 10 players or fewer, and 13 have 0 (section 8). That is my hand count. |
| Why it matters | A difference between 1 and 2 players is noise. If we rank or average all games, the small ones add a lot of noise and show nothing real. |
| What dbt does | In the marts, do not compare all games in the same way. Three options: (a) only compare games above a minimum number of players, (b) only show the top N games, (c) show a log scale in the dashboard. |
| Decision needed | The minimum is not decided. Aira to choose after we have more snapshots. |
| Layer | Marts |

---

### Finding 6: a count is one moment

| | |
|---|---|
| Findings | One run takes about 10 minutes (section 1). Stellaris went from 7,859 to 8,106 in one morning (section 7). |
| Why it matters | Player numbers change during the day. If we run at a different time every day, differences between days may just be time-of-day effects. |
| What to do | Start the daily run at about the same clock time. Say this under limits in the thesis. In the marts, show the hour of the snapshot next to the count (`collected_hour_local` from finding 2). |
| Layer | Process (how we run it), marts (the column) |

---

### Finding 7: history

| | |
|---|---|
| Findings | There is one snapshot of player counts so far. Each daily run adds 81 rows. |
| Why it matters | Several rows for the same game are not duplicates. They are the history we want for the trend charts. |
| What dbt does | Nothing special in staging. Keep all rows. For "players now" questions, use the latest row per game (same pattern as finding 4). For trends, use all rows. |
| Layer | n/a |

---

**Still open**

| Question | Status |
|---|---|
| Can a released game ever get 404 / `result = 42`? What does 42 mean exactly? | Not seen yet, and 42 is unverified |
| Minimum player count for comparisons | Not decided |
| Why does Knights of Pen and Paper 2: Free Edition have no reviews and 0 players? | Unknown |
| Players against reviews and price | No correlation calculated, needs more snapshots |
| Rule for repeated runs (one run per day?) | Proposal, not decided. Finding 2 depends on it. |
| Is `dbt_utils` installed? | Not checked |

In [26]:
# conn.close()